H1.a

In [2]:
import duckdb

# only reads these 3 columns from disk, regardless of how many columns the parquet has
a2t = duckdb.sql("""
    SELECT authors, year, gini_field, share_ai_7y, quant_prod, quant_cite, cohort, gender, period, prod_7y, cite_7y
    FROM 'a2t_final.parquet'
""").df()

BinderException: Binder Error: Referenced column "gender" not found in FROM clause!
Candidate bindings: "field_energy", "field_veterinary", "field_neuroscience", "field_computer_science", "field_materials_science"

In [ ]:
import pandas as pd
def safe_qcut(s, q=5):
    binned = pd.qcut(s, q, duplicates='drop')
    n_bins = binned.cat.categories.size
    return pd.qcut(s, q, labels=range(1, n_bins + 1), duplicates='drop')

a2t["cite_cor"] = a2t['cite_7y']/a2t['prod_7y']

a2t["quant_cite_cor"] = a2t.groupby("period")["cite_cor"].transform(safe_qcut)


In [1]:
counts = a2t.groupby(["year","cohort","quant_cite_cor", "gender"])["authors"].nunique().reset_index(name="count")

NameError: name 'a2t' is not defined

In [ ]:
counts.to_csv("cite_cor_counts.csv",index=False)

In [ ]:
import pandas as pd
counts = pd.read_csv("cite_counts.csv")

In [ ]:
pivot = counts.pivot_table(
    index=["year", "cohort", "quant_cite_cor"],
    columns="gender",
    values="count",
    fill_value=0
).reset_index()

pivot["total"] = pivot["F"] + pivot["M"]
pivot["share_women"] = pivot["F"] / pivot["total"]

# 2. Aggregate share per year+cohort (across quantiles)
agg = pivot.groupby(["year", "cohort"])[["F", "M"]].sum().reset_index()
agg["total_yc"] = agg["F"] + agg["M"]
agg["share_women_yc"] = agg["F"] / agg["total_yc"]

# 3. Merge back and compute ratio
pivot = pivot.merge(agg[["year", "cohort", "share_women_yc"]], on=["year", "cohort"], how="left")
pivot["share_ratio"] = pivot["share_women"] / pivot["share_women_yc"]

Graphs

In [ ]:
# prod
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8,5))
for cohort, g in pivot.groupby("cohort"):  # agg_full = df with share_ratio computed
    g_sorted = g.groupby("quant_prod")["share_ratio"].mean().reset_index()
    ax.plot(g_sorted["quant_prod"], g_sorted["share_ratio"], marker="o", label=cohort)

ax.axhline(1, color="black", linestyle="--", linewidth=1)  # reference: equal representation
ax.set_xlabel("Quantile (prod)")
ax.set_xticks(range(1,6))
ax.set_ylabel("Share ratio (women, relative to cohort average)")
ax.legend(title="Cohort")
plt.show()

Annex

In [ ]:
# plot first
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.8 / pivot["cohort"].nunique()
pivot["first_quant_prod"] = pivot["first_quant_prod"].astype(int)
for i, (cohort, g) in enumerate(pivot.groupby("cohort")):
    g_sorted = g.groupby("first_quant")["share_ratio"].mean().reset_index().sort_values("first_quant_prod")
    x = g_sorted["first_quant_prod"] + i * width
    ax.bar(x, g_sorted["share_ratio"], width=width, label=cohort)

ax.axhline(1, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Quantile (prod)")
ax.set_ylabel("Share ratio (women, relative to cohort average)")
ax.legend(title="Cohort")
plt.show()

H1.b

In [ ]:
# transition matrix prod
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from itertools import product

genders = ["M", "F"]
periods = range(1, 8)

fig, axes = plt.subplots(len(genders), len(periods), figsize=(4*len(periods), 4*len(genders)))

for row, g in enumerate(genders):
    for col, i in enumerate(periods):
        ax = axes[row, col]
        trans_matrix = pd.crosstab(
            a2t[(a2t["gender"]==g)&(a2t["period"]==i)]["prev_quant_prod"],
            a2t[(a2t["gender"]==g)&(a2t["period"]==i)]["quant_prod"]
        )

        sns.heatmap(trans_matrix, annot=True, fmt=".2f", cmap="crest", ax=ax, vmin=0, cbar=False)

        ax.set_xlabel("Quantile (current)")
        ax.set_ylabel("Quantile (previous)" if col == 0 else "")
        ax.set_title(f"{g} - period {i}")

plt.tight_layout()
plt.show()

In [ ]:
import duckdb

# only reads these 3 columns from disk, regardless of how many columns the parquet has
a2t = duckdb.sql("""
    SELECT authors,cohort,gender,career_age,insti_rank,first_quant_prod,gini_field, year, period, quant_prod, share_ai_7y, prev_coaut, quant_cite, first_quant_cite, log_prod, prev_quant_prod, prev_quant_cite, prev_prod_7y
    FROM 'a2t_final.parquet'
""").df()

In [ ]:
a2t = a2t.sort_values(["cohort", "authors", "year"]).copy()

# Unique author/cohort/year combinations
years = (
    a2t[["cohort", "authors", "year"]]
    .drop_duplicates()
    .sort_values(["cohort", "authors", "year"])
)

# Previous YEAR for each author/cohort
years["start"] = (
    years.groupby(["cohort", "authors"])["year"]
         .shift(1)
)

# First observation starts one year before
years["start"] = years["start"].fillna(
    years.groupby(["cohort", "authors"])["year"].transform("min") - 1
)

# Merge the interval back onto every article
a2t = a2t.drop(columns=["start", "stop"], errors="ignore")

a2t = a2t.merge(
    years,
    on=["cohort", "authors", "year"],
    how="left"
)

a2t["stop"] = a2t["year"]

In [ ]:
import pandas as pd
a2t = pd.get_dummies(a2t, columns=["gender"], drop_first=True)
a2t["cohort"] = a2t["cohort"].astype("category")
a2t["insti_rank"] = a2t["insti_rank"].astype("category")
a2t["first_quant_prod"] = a2t["first_quant_prod"].astype("category")
a2t["first_quant_cite"] = a2t["first_quant_cite"].astype("category")

In [ ]:
# QX → QY
#a2t["event"] = ((a2t["quant_prod"]==5) & (a2t["prev_quantile"]==4)).astype(int)

a2t.dropna(subset=["prev_quant_prod"], inplace=True)

# +1 quantile
a2t["event"] = (a2t["quant_cite"]>a2t["prev_quant_cite"]).astype(int)

In [ ]:
# prod 
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t.dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + gender_M + first_prod + gini_field + share_3 + insti_rank"
)
ctv.print_summary()

In [ ]:
# cite
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","first_cite","gini_field","share_3","log_prod"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + insti_rank + gender_M + first_cite + gini_field + share_3 + log_prod + log_prod * gender_M"
)
ctv.print_summary()

Annex

In [ ]:
# prod check
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t,#[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","first_prod","gini_field","share_3","year"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort", "first_quant_prod", "insti_rank"],
    formula="gender_M * career_age  + gini_field + share_ai_7y + share_ai_7y:year + career_age:year + gini_field:year"
)
ctv.print_summary()
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t,#[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","first_prod","gini_field","share_3","year"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort", "first_quant_prod", "insti_rank"],
    formula="gender_M * career_age  + gini_field + share_ai_7y")
ctv.print_summary()

H1.c

In [ ]:
import duckdb

# only reads these 3 columns from disk, regardless of how many columns the parquet has
a2t = duckdb.sql("""
    SELECT authors,cohort,gender,career_age,insti_rank,first_quant_prod,gini_field, year, period, quant_prod, share_ai_7y, prev_coaut, quant_cite, first_quant_cite, log_prod, prev_quant_prod, prev_quant_cite, prev_prod_7y
    FROM 'a2t_final.parquet'
""").df()

In [ ]:
a2t = a2t.sort_values(["cohort", "authors", "year"]).copy()

# Unique author/cohort/year combinations
years = (
    a2t[["cohort", "authors", "year"]]
    .drop_duplicates()
    .sort_values(["cohort", "authors", "year"])
)

# Previous YEAR for each author/cohort
years["start"] = (
    years.groupby(["cohort", "authors"])["year"]
         .shift(1)
)

# First observation starts one year before
years["start"] = years["start"].fillna(
    years.groupby(["cohort", "authors"])["year"].transform("min") - 1
)

# Merge the interval back onto every article
a2t = a2t.drop(columns=["start", "stop"], errors="ignore")

a2t = a2t.merge(
    years,
    on=["cohort", "authors", "year"],
    how="left"
)

a2t["stop"] = a2t["year"]

In [ ]:
a2t = a2t.sort_values(["cohort", "authors", "year"]).reset_index(drop=True)

a2t["prev_year"] = (
    a2t.groupby(["cohort", "authors"])["year"].shift(1)
)

a2t = a2t.dropna(subset=["prev_year"])

In [ ]:
import pandas as pd
a2t = pd.get_dummies(a2t, columns=["gender"], drop_first=True)
a2t["cohort"] = a2t["cohort"].astype("category")
a2t["insti_rank"] = a2t["insti_rank"].astype("category")
a2t["first_quant_prod"] = a2t["first_quant_prod"].astype("category")
a2t["first_quant_cite"] = a2t["first_quant_cite"].astype("category")

In [ ]:
# leave QX
a2t["event"] = ((a2t["quant_prod"]!=5)&(a2t["prev_quantile"]==5)&(a2t["year"]-a2t["prev_year"]<6)).astype(int)  # or keep first rows if entry year defined

# X-year interruption
#a2t["event"] = (a2t["year"]-a2t["prev_year"]>3).astype(int)

In [ ]:
# prod

from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","share_3","gini_field","first_prod"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + insti_rank + gender_M + gini_field + share_3 + first_prod"
)
ctv.print_summary()

In [ ]:
# cite
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","first_cite","gini_field","share_3","log_prod"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + insti_rank + gender_M + first_cite + gini_field + share_3 + log_prod + log_prod * gender_M"
)
ctv.print_summary()

In [ ]:
# 5 year interruption, first_prod
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","share_3","gini_field","first_prod"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + insti_rank + gender_M + gini_field + share_3 + first_prod"
)
ctv.print_summary()

In [ ]:
# 5 year interruption, first_cite
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","first_cite","gini_field","share_3","log_prod"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + insti_rank + gender_M + first_cite + gini_field + share_3 + log_prod + log_prod * gender_M"
)
ctv.print_summary()

Annex

In [ ]:
import duckdb

# note: this only works if the parquet has been read into a table/view first,
# since information_schema/duckdb_columns() need a registered relation, not a raw file path
duckdb.sql("CREATE OR REPLACE VIEW a2t AS SELECT * EXCLUDE(field_computer_science) FROM 'a2t_final.parquet'")

cohorts = duckdb.sql("""
    SELECT DISTINCT cohort
    FROM a2t
""").df()["cohort"].tolist()

In [ ]:
import duckdb

def proba_cond(gender, Q, quant, cohort=None):
    con = duckdb.connect()
    con.execute("PRAGMA disable_progress_bar")

    cohort_filter = f"AND cohort = '{cohort}'" if cohort else ""

    query = f"""
        WITH author_quant AS (
            SELECT
                authors,
                gender,
                cohort,
                FIRST({quant} ORDER BY year) AS q_value
            FROM read_parquet('a2t_dec_final.parquet')
            WHERE gender = '{gender}' {cohort_filter}
            GROUP BY authors, gender, cohort
        )
        SELECT
            COUNT(*) FILTER (WHERE q_value = {Q}) AS A,
            COUNT(*) AS B
        FROM author_quant
    """
    res = con.execute(query).df()
    A, B = res.iloc[0]["A"], res.iloc[0]["B"]
    con.close()
    return A / B if B else None

In [ ]:
import duckdb
import pandas as pd
records = []
for quant in ["quant_prod", "quant_cite"]:
    for Q in range(1,6):
        for g in ["M", "F"]:
            for cohort in cohorts:
                records.append({
                    "quant": quant,
                    "Q": Q,
                    "gender": g,
                    "cohort": cohort,
                    "proba": proba_cond(g, Q, quant, cohort),
                })

df = pd.DataFrame(records)
df.head()

In [ ]:
df.to_csv("prob_cond.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt

colors = {"M": "tab:blue", "F": "tab:red"}

quants = df["quant"].unique()
Qs = df["Q"].unique()

n_rows = len(Qs)
n_cols = len(quants)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows), squeeze=False)

for i, quant in enumerate(quants):
    for j, Q in enumerate(Qs):
        ax = axes[j][i]
        subset = df[(df["quant"] == quant) & (df["Q"] == Q)]

        for g in ["M", "F"]:
            sub_g = subset[subset["gender"] == g].sort_values("cohort")
            ax.plot(
                sub_g["cohort"], sub_g["proba"],
                marker="o", label=g, color=colors[g]
            )

        ax.set_title(f"P(Q{Q} | genre, cohorte) — {quant}")
        ax.set_xlabel("Cohorte")
        ax.set_ylabel(f"Probabilité d'être dans Q{Q}")
        ax.legend(title="Genre")
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 3 year interruption, first_prod
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","share_3","gini_field","first_prod"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + insti_rank + gender_M + gini_field + share_3 + first_prod"
)
ctv.print_summary()

In [ ]:
# 3 year interruption, first_cite
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(
    a2t[["authors","cohort","start","stop","event","gender_M","career_age","insti_rank","first_cite","gini_field","share_3","log_prod"]].dropna(),
    id_col='authors',
    start_col='start',
    stop_col='stop',
    event_col='event',
    strata=["cohort"],
    formula="gender_M * career_age + career_age + insti_rank + gender_M + first_cite + gini_field + share_3 + log_prod + log_prod * gender_M"
)
ctv.print_summary()